In [ ]:
import sys
sys.path.append("../")

import torch

from npu_yolov8n import load_NPU_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
checkpoint_path = '../checkpoints/qat_fixed.pt' 
loaded_cfg = torch.load(checkpoint_path, device, weights_only=False)['qcfg']
Nmodel = load_NPU_model(checkpoint_path, device, loaded_cfg)


In [ ]:
layer_type_map = {
    '0': 'Conv',
    '1': 'Conv',
    '2': 'C2f',
    '3': 'Conv',
    '4': 'C2f',
    '5': 'Conv',
    '6': 'C2f',
    '7': 'Conv',
    '8': 'C2f',
    '9': 'SPPF',
    '10': 'Upsample',
    '11': 'Concat',
    '12': 'C2f',
    '13': 'Upsample',
    '14': 'Concat',
    '15': 'C2f',
    '16': 'Conv',
    '17': 'Concat',
    '18': 'C2f',
    '19': 'Conv',
    '20': 'Concat',
    '21': 'C2f',
    '22': 'Detect'
}

def serialize_cfg(cfg, layer_type_map):
    serialized = {}
    for k, v in cfg.items():
        layer_idx = k.split('_')[1]
        layer_type = layer_type_map.get(layer_idx, "unknown")
        if layer_type == "Conv":
            serialized[f"model.{layer_idx}.conv"] = v
        elif layer_type == "C2f":
            serialized[f"model.{layer_idx}.cv1.conv"] = v['cv1']
            serialized[f"model.{layer_idx}.cv2.conv"] = v['cv2']
            n=0
            while f'm.{n}' in v:
                serialized[f"model.{layer_idx}.m.{n}.cv1.conv"] = v[f'm.{n}']['cv1']
                serialized[f"model.{layer_idx}.m.{n}.cv2.conv"] = v[f'm.{n}']['cv2']
                n += 1
        elif layer_type == "SPPF":
            serialized[f"model.{layer_idx}.cv1.conv"] = v['cv1']
            serialized[f"model.{layer_idx}.cv1.conv"] = v['cv1']
        elif layer_type == "Detect":
            serialized[f"model.{layer_idx}.cv2.0.0.conv"] = v['cv2.0']
            serialized[f"model.{layer_idx}.cv2.1.0.conv"] = v['cv2.0']
            serialized[f"model.{layer_idx}.cv2.2.0.conv"] = v['cv2.0']

            serialized[f"model.{layer_idx}.cv2.0.1.conv"] = v['cv2.1']
            serialized[f"model.{layer_idx}.cv2.1.1.conv"] = v['cv2.1']
            serialized[f"model.{layer_idx}.cv2.2.1.conv"] = v['cv2.1']

            serialized[f"model.{layer_idx}.cv2.0.2"] = v['cv2.2']
            serialized[f"model.{layer_idx}.cv2.1.2"] = v['cv2.2']
            serialized[f"model.{layer_idx}.cv2.2.2"] = v['cv2.2']

            serialized[f"model.{layer_idx}.cv3.0.0.conv"] = v['cv3.0']
            serialized[f"model.{layer_idx}.cv3.1.0.conv"] = v['cv3.0']
            serialized[f"model.{layer_idx}.cv3.2.0.conv"] = v['cv3.0']

            serialized[f"model.{layer_idx}.cv3.0.1.conv"] = v['cv3.1']
            serialized[f"model.{layer_idx}.cv3.1.1.conv"] = v['cv3.1']
            serialized[f"model.{layer_idx}.cv3.2.1.conv"] = v['cv3.1']

            serialized[f"model.{layer_idx}.cv3.0.2"] = v['cv3.2']
            serialized[f"model.{layer_idx}.cv3.1.2"] = v['cv3.2']
            serialized[f"model.{layer_idx}.cv3.2.2"] = v['cv3.2']
        else:
            print(f"--- Warning: layer {layer_idx} type is unknown ---")

    return serialized

In [ ]:
ncfg = Nmodel.ncfg
scfg = serialize_cfg(ncfg, layer_type_map)
scfg

In [ ]:
import struct, torch, numpy as np
from unittest import case

meta_bin = open("qmeta.bin","wb")
weight_bin = open("qweights.bin","wb")

entry_count = 0
entries = [] 

for name, tensor in Nmodel.state_dict().items():

    # layer = name.split('.')[1]
    layer = int(name.split('.')[1])
    dtype = 1 if "weight" in name else 2     # 1=int8 , 2=int16
    # get shift from ncfg
    for k, v in scfg.items():
        if k in name:
            shift = v['shift']

    arr = tensor.cpu().numpy()
    # q = arr.astype(np.int8 if dtype==1 else np.int16)
    q = arr.astype(np.int32)

    offset = weight_bin.tell()
    weight_bin.write(q.tobytes())

    shape = list(arr.shape)
    shape += [0]*(4-len(shape))  # shape is fixed to 4D

    ltype = layer_type_map[name.split('.')[1]]
    if ltype == 'Conv':
        layer_type = 0  
    elif ltype == 'C2f':
        layer_type = 1
    elif ltype == 'SPPF':
        layer_type = 2
    elif ltype == 'Detect':
        layer_type = 3
    else: 
        layer_type = 4  # unknown
    entries.append((name,layer,layer_type,dtype,shift,offset,q.size,*shape))
    if 'weight' in name:
        print(len(entries), name, shape, shift)

entry_count = len(entries)

# ---- metadata binary ----
meta_bin.write(struct.pack("I", entry_count))

for e in entries:
    name, layer, layer_type, dtype, shift, offset, size, s0,s1,s2,s3 = e

    meta_bin.write(struct.pack("32s", name.encode()))
    meta_bin.write(struct.pack("B", layer))

    meta_bin.write(struct.pack("B", layer_type))
    meta_bin.write(struct.pack("B", dtype))    # 1=int8,2=int16
    meta_bin.write(struct.pack("B", shift))    # 0~15
    meta_bin.write(struct.pack("I", offset))
    meta_bin.write(struct.pack("I", size))
    meta_bin.write(struct.pack("IIII", s0,s1,s2,s3))

meta_bin.close()
weight_bin.close()

print("Saved qmeta.bin + qweights.bin")


In [ ]:
import cv2
import matplotlib.pyplot as plt

img = cv2.imread("./test_images/0th__predict.jpg")
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.imshow(img_rgb)
plt.axis('off')
plt.show()


In [ ]:
type(img_rgb)

In [ ]:
from npu_yolov8n.models.blocks.qat_blocks import custom_floor_clip

def quantize_input(x):
    # input quantization
    input_precision = 8
    input_fraction_bits = 7
    input_scale = 2 ** input_fraction_bits
    input_min = -2 ** ( input_precision - 1)
    input_max = 2 ** (input_precision - 1) - 1
    x = custom_floor_clip(x*input_scale, input_min, input_max) # convert to int8 range
    return x



In [ ]:
x_quant = quantize_input(torch.from_numpy(img_rgb/255).permute(2, 0, 1))
x_quant = x_quant.numpy().astype(np.int32)
x_quant.tofile("image.bin")     
